In [1]:
import os
import random
import json
import time
import zipfile
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# ==========================================
# 1. Setup, Configuration & Authentication
# ==========================================
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

PROPORTIONS = {
    "train_supreme_court.json": 0.40,
    "train_high_courts.json": 0.25,
    "train_acts.json": 0.20,
    "train_tribunals.json": 0.15
}

# Hyperparameters
TRAIN_TOKENS = 100_000_000  # Adjust based on your time constraints
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)

# Custom Model Dimensions (~50M params)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 15
N_HEAD = 8
N_KV_HEAD = 4 
VOCAB_SIZE = 32000

# Training Optimizations (2x T4 specific)
BATCH_SIZE = 2       # Scaled up for dual GPUs
ACCUMULATION_STEPS = 16 # Effective batch size = 32

# Generation Config
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.2
TOP_P = 0.85

# Paths
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-custom-final")
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "latest_checkpoint.pt")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU count: 2


In [3]:
data_files = {}
for filename in PROPORTIONS.keys():
    print(f"Downloading {filename}...")
    # The training files are located in the Phase_1_Pretraining/Train/ directory
    file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=f"Phase_1_Pretraining/Train/{filename}",
        repo_type="dataset",
        token=HF_TOKEN
    )
    data_files[filename] = file_path

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

In [8]:
for filename,filepath in data_files.items():
    print(f"{filename} - > {filepath}")

train_supreme_court.json - > /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
train_high_courts.json - > /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
train_acts.json - > /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
train_tribunals.json - > /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


In [4]:
import json
import os
import random
from collections import Counter, defaultdict

class CustomScratchBPE:
    def __init__(self, vocab_size, special_tokens=None):
        self.vocab_size = vocab_size
        self.special_tokens = special_tokens or ["<PAD>","<UNK>", "<BOS>", "<EOS>"]
        self.vocab = {}       
        self.inv_vocab = {}   
        self.merges = {}      
        
    def train(self, text_list):
        print("Aggregating and filtering word frequencies...")
        word_freqs = Counter()
        
        for text in text_list:
            words = ["Ġ" + w for w in text.split() if w and len(w) <= 35]
            word_freqs.update(words)
            
        # Drop singletons to save RAM
        word_freqs = Counter({word: freq for word, freq in word_freqs.items() if freq > 1})
        print(f"Unique words after filtering singletons: {len(word_freqs):,}")
            
        unique_chars = set()
        for word in word_freqs:
            unique_chars.update(list(word))
            
        idx = 0
        for st in self.special_tokens:
            self.vocab[st] = idx
            self.inv_vocab[idx] = st
            idx += 1
            
        for ch in sorted(list(unique_chars)):
            self.vocab[ch] = idx
            self.inv_vocab[idx] = ch
            idx += 1
            
        splits = {word: [self.vocab[c] for c in word] for word in word_freqs}
        target_merges = self.vocab_size - len(self.vocab)
        print(f"Base character vocab size: {len(self.vocab)}. Target merges: {target_merges}")
        
        # ---------------------------------------------------------
        # SPEED OPTIMIZATION: Initial Pair Counts & Inverted Index
        # ---------------------------------------------------------
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)
        
        for word, freq in word_freqs.items():
            split = splits[word]
            for j in range(len(split) - 1):
                pair = (split[j], split[j+1])
                pair_counts[pair] += freq
                pair_to_words[pair].add(word)

        for i in range(target_merges):
            if not pair_counts:
                break
                
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = idx
            idx += 1
            
            self.merges[best_pair] = new_id
            new_token_str = self.inv_vocab[best_pair[0]] + self.inv_vocab[best_pair[1]]
            self.vocab[new_token_str] = new_id
            self.inv_vocab[new_id] = new_token_str
            
            # ---------------------------------------------------------
            # FAST INCREMENTAL UPDATE: Only modify words containing best_pair
            # ---------------------------------------------------------
            words_to_update = list(pair_to_words.get(best_pair, []))
            
            del pair_counts[best_pair]
            if best_pair in pair_to_words:
                del pair_to_words[best_pair]
            
            for word in words_to_update:
                split = splits[word]
                freq = word_freqs[word]
                
                # 1. Remove old pairs from global tracking for this word
                for j in range(len(split) - 1):
                    pair = (split[j], split[j+1])
                    if pair != best_pair:
                        pair_counts[pair] -= freq
                        if pair_counts[pair] <= 0:
                            del pair_counts[pair]
                        if word in pair_to_words[pair]:
                            pair_to_words[pair].remove(word)

                # 2. Perform the merge on this word's split
                new_split = []
                j = 0
                while j < len(split):
                    if j < len(split) - 1 and (split[j], split[j+1]) == best_pair:
                        new_split.append(new_id)
                        j += 2
                    else:
                        new_split.append(split[j])
                        j += 1
                splits[word] = new_split

                # 3. Add newly created pairs to global tracking for this word
                for j in range(len(new_split) - 1):
                    pair = (new_split[j], new_split[j+1])
                    pair_counts[pair] += freq
                    pair_to_words[pair].add(word)
                
            if (i + 1) % 500 == 0:
                print(f"Completed {i + 1}/{target_merges} merges...")

    def encode(self, text):
        words = ["Ġ" + w for w in text.split() if w and len(w) <= 35]
        tokens = []
        unk_id = self.vocab.get("[UNK]", 0)
        
        for word in words:
            split = [self.vocab.get(c, unk_id) for c in word]
            while len(split) > 1:
                min_idx = -1
                min_rank = float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair]
                        if rank < min_rank:
                            min_rank = rank
                            min_idx = i
                if min_idx == -1:
                    break
                pair = (split[min_idx], split[min_idx+1])
                new_id = self.merges[pair]
                split = split[:min_idx] + [new_id] + split[min_idx+2:]
            tokens.extend(split)
        return tokens

    def decode(self, token_ids):
        text = "".join(self.inv_vocab.get(t, "[UNK]") for t in token_ids)
        return text.replace("Ġ", " ").strip()

    def get_vocab_size(self):
        return len(self.vocab)

    def save(self, path):
        serializable_merges = {f"{k[0]},{k[1]}": v for k, v in self.merges.items()}
        data = {"vocab": self.vocab, "merges": serializable_merges, "special_tokens": self.special_tokens}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.vocab = data["vocab"]
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.special_tokens = data["special_tokens"]
        self.merges = {tuple(map(int, k.split(","))): v for k, v in data["merges"].items()}


# ==========================================
# 2. Load or Train BPE Tokenizer
# ==========================================
tokenizer_path = os.path.join(OUTPUT_DIR, "custom_bpe_scratch.json")

tokenizer = CustomScratchBPE(vocab_size=32000)

if os.path.exists(tokenizer_path):
    print(f"Loading existing tokenizer from {tokenizer_path}...")
    tokenizer.load(tokenizer_path)
    print(f"Loaded Vocabulary size: {tokenizer.get_vocab_size()}")
else:
    print("Streaming subset for tokenizer training...")
    text_corpus_for_tokenizer = []

    for filename, filepath in data_files.items():
        print(f"Adding from {filename}")
        records = []
        
        with open(filepath, "r", encoding="utf-8") as f:
            first_char = f.read(1)
            f.seek(0)
            
            if first_char != '[':
                for i, line in enumerate(f):
                    if i > 5000: break
                    try:
                        obj = json.loads(line)
                        if isinstance(obj, dict) and "text" in obj:
                            records.append(obj)
                    except:
                        continue
            else:
                for line in f:
                    if len(records) >= 2000: break
                    try:
                        obj = json.loads(line.strip().strip(','))
                        if isinstance(obj, dict) and "text" in obj:
                            records.append(obj)
                    except:
                        continue

        if records:
            sample_docs = random.sample(records, min(1000, len(records)))
            text_corpus_for_tokenizer.extend([d["text"] for d in sample_docs if "text" in d])

    print("Training Custom BPE Tokenizer from scratch (Optimized Incremental)...")
    tokenizer.train(text_corpus_for_tokenizer)
    tokenizer.save(tokenizer_path)
    print(f"Final Vocabulary size: {tokenizer.get_vocab_size()}")

Loading existing tokenizer from /kaggle/working/cs781_phase1/custom_bpe_scratch.json...
Loaded Vocabulary size: 32000


In [5]:
# ==========================================
# 3. Tokenization & Dataset Creation (Streaming & Zero-Memory Overhead)
# ==========================================
print("Processing and mixing tokens via streaming...")

# Pre-allocate 1D PyTorch Tensors for training and validation tokens
train_tokens_tensor = torch.zeros(TRAIN_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long,device='cpu')
val_tokens_tensor = torch.zeros(VAL_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long,device='cpu')

current_train = 0
current_val = 0

for filename, filepath in data_files.items():
    weight = PROPORTIONS[filename]
    target_train = int(TRAIN_TOKENS * weight)
    target_val = int(VAL_TOKENS * weight)
    file_train = 0
    file_val = 0
    
    print(f"Tokenizing and streaming from {filename}...")
    with open(filepath, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)
        
        # Stream file line-by-line depending on format to protect RAM
        for i, line in enumerate(f):
            if i % 15000 == 0 and i > 0:
                print(f"[{filename}] Processed {i:,} lines... (Train: {file_train:,} | Val: {file_val:,})")

            if file_train >= target_train and file_val >= target_val:
                break
                
            try:
                if first_char != '[':
                    data = json.loads(line)
                else:
                    # Clean commas for line-by-line JSON array parsing
                    stripped = line.strip().rstrip(',')
                    if stripped in ('[', ']'):
                        continue
                    data = json.loads(stripped)
            except:
                continue
                
            if not isinstance(data, dict) or "text" not in data:
                continue
                
            tokens = tokenizer.encode(data["text"])
            if not tokens:
                continue
                
            tok_tensor = torch.tensor(tokens, dtype=torch.long)
            tok_len = len(tok_tensor)
            
            # Randomly split into validation (5%) or training (95%)
            if file_train >= target_train:
                is_val = True  # Train is full, force into val
            elif file_val >= target_val:
                is_val = False # Val is full, force into train
            else:
                is_val = random.random() < 0.05 # Normal 5% split

            # Append based on the smart routing
            # Append based on the smart routing using exact slicing
            if is_val:
                # Calculate exactly how much space is left for this file and global tensor
                space_left = min(target_val - file_val, len(val_tokens_tensor) - current_val)
                if space_left > 0:
                    take_len = min(tok_len, space_left)
                    val_tokens_tensor[current_val : current_val + take_len] = tok_tensor[:take_len]
                    current_val += take_len
                    file_val += take_len
            else:
                # Calculate exactly how much space is left for this file and global tensor
                space_left = min(target_train - file_train, len(train_tokens_tensor) - current_train)
                if space_left > 0:
                    take_len = min(tok_len, space_left)
                    train_tokens_tensor[current_train : current_train + take_len] = tok_tensor[:take_len]
                    current_train += take_len
                    file_train += take_len
                    
    print(f"{filename}: Collected {file_train:,} train tokens and {file_val:,} val tokens.")

# Trim tensors to exact block boundaries and reshape using .view() (zero data copying)
train_len = (current_train // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE
val_len = (current_val // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE

train_blocks = train_tokens_tensor[:train_len].view(-1, TRAIN_BLOCK_SIZE)
val_blocks = val_tokens_tensor[:val_len].view(-1, TRAIN_BLOCK_SIZE)

print(f"Total Training Blocks: {len(train_blocks):,} | Total Validation Blocks: {len(val_blocks):,}")

class BlockDataset(torch.utils.data.Dataset):
    def __init__(self, blocks):
        self.blocks = blocks
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, idx):
        return self.blocks[idx, :-1], self.blocks[idx, 1:]

Processing and mixing tokens via streaming...
Tokenizing and streaming from train_supreme_court.json...
train_supreme_court.json: Collected 40,000,000 train tokens and 2,000,000 val tokens.
Tokenizing and streaming from train_high_courts.json...
[train_high_courts.json] Processed 15,000 lines... (Train: 6,835,281 | Val: 350,287)
[train_high_courts.json] Processed 30,000 lines... (Train: 13,668,692 | Val: 719,908)
[train_high_courts.json] Processed 45,000 lines... (Train: 20,305,993 | Val: 1,080,267)
train_high_courts.json: Collected 25,000,000 train tokens and 1,250,000 val tokens.
Tokenizing and streaming from train_acts.json...
train_acts.json: Collected 20,000,000 train tokens and 1,000,000 val tokens.
Tokenizing and streaming from train_tribunals.json...
train_tribunals.json: Collected 15,000,000 train tokens and 750,000 val tokens.
Total Training Blocks: 48,828 | Total Validation Blocks: 2,441


In [6]:

train_loader = torch.utils.data.DataLoader(
    BlockDataset(train_blocks), 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    pin_memory=True, 
    num_workers=0
)
val_loader = torch.utils.data.DataLoader(
    BlockDataset(val_blocks), 
    batch_size=BATCH_SIZE, 
    pin_memory=True
)

print("DataLoaders successfully created!")
print(BATCH_SIZE)

DataLoaders successfully created!
2


In [7]:
import math
import torch
import torch.nn as torch_nn

class ManualTransformerAttention(torch_nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        self.scale = 1.0 / math.sqrt(head_dim)

    # Added enc_mask to the function signature
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, is_causal: bool = True, enc_mask: torch.Tensor = None):
        B_q, H, L_q, D = q.shape
        _, _, L_k, _ = k.shape
        
        k_transposed = k.transpose(-2, -1).contiguous()
        scores = torch.matmul(q, k_transposed) * self.scale
        
        if is_causal and L_q > 1:
            mask = torch.tril(torch.ones(L_q, L_k, device=q.device)).view(1, 1, L_q, L_k)
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        if enc_mask is not None: # Now correctly references the passed parameter
            scores = scores + enc_mask.to(scores.dtype)
            
        weights = torch.softmax(scores, dim=-1)
        out = torch.matmul(weights, v.contiguous())
        return out

In [8]:
# ==========================================
# 1. Custom Mathematical Components
# ==========================================
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    # CRITICAL FIX: Protect buffer from DataParallel corruption by storing it as real
    return torch.view_as_real(freqs_cis)

def apply_rope_single(x: torch.Tensor, freqs_cis: torch.Tensor):
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, x.shape[1], 1, x_complex.shape[-1])
    x_rotated = torch.view_as_real(x_complex * freqs_cis).flatten(3)
    return x_rotated.type_as(x)

class RMSNorm(torch_nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = torch_nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class SwiGLU(torch_nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = torch_nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = torch_nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = torch_nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

# ==========================================
# 2. Universal Manual Attention (With KV Cache)
# ==========================================
class ManualGQA(torch_nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        
        self.wq = torch_nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = torch_nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = torch_nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = torch_nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        
        # Instantiate your custom attention class
        self.core_attention = ManualTransformerAttention(self.head_dim)
        
    def forward(self, q_x, kv_x=None, freqs_cis_q=None, freqs_cis_k=None, is_causal=False, enc_mask=None, past_kv=None):
        B, T_q, C = q_x.shape
        xq = self.wq(q_x).view(B, T_q, self.n_heads, self.head_dim)
        
        # --- KV Cache State Management ---
        if past_kv is not None:
            cached_k, cached_v = past_kv
            if kv_x is not None: 
                T_k = kv_x.shape[1]
                xk = self.wk(kv_x).view(B, T_k, self.n_kv_heads, self.head_dim)
                xv = self.wv(kv_x).view(B, T_k, self.n_kv_heads, self.head_dim)
                if freqs_cis_k is not None:
                    xk = apply_rope_single(xk, freqs_cis_k)
                xk = torch.cat([cached_k, xk], dim=1)
                xv = torch.cat([cached_v, xv], dim=1)
            else:
                xk, xv = cached_k, cached_v
        else:
            T_k = kv_x.shape[1]
            xk = self.wk(kv_x).view(B, T_k, self.n_kv_heads, self.head_dim)
            xv = self.wv(kv_x).view(B, T_k, self.n_kv_heads, self.head_dim)
            if freqs_cis_k is not None:
                xk = apply_rope_single(xk, freqs_cis_k)
                
        current_kv = (xk, xv)

        if freqs_cis_q is not None:
            xq = apply_rope_single(xq, freqs_cis_q)

        xk_rep = xk.repeat_interleave(self.n_rep, dim=2)
        xv_rep = xv.repeat_interleave(self.n_rep, dim=2)
        
        q = xq.transpose(1, 2)
        k = xk_rep.transpose(1, 2)
        v = xv_rep.transpose(1, 2)
        
        # Call the dedicated attention logic and pass the enc_mask
        out = self.core_attention(q, k, v, is_causal=is_causal, enc_mask=enc_mask)
        
        out = out.transpose(1, 2).contiguous().view(B, T_q, C)
        return self.wo(out), current_kv

# ==========================================
# 3. Model Blocks
# ==========================================
class EncoderBlock(torch_nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, hidden_dim):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = ManualGQA(d_model, n_heads, n_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, hidden_dim)

    def forward(self, x, freqs_cis, enc_mask=None):
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), freqs_cis_q=freqs_cis, freqs_cis_k=freqs_cis, enc_mask=enc_mask)
        h = x + attn_out
        return h + self.ffn(self.norm2(h))

class DecoderBlock(torch_nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, hidden_dim):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = ManualGQA(d_model, n_heads, n_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = ManualGQA(d_model, n_heads, n_kv_heads)
        self.norm3 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, hidden_dim)#from LlaMa

    def forward(self, x, enc_out, freqs_cis_dec, freqs_cis_enc, enc_mask=None, past_kvs=None):
        self_kv = past_kvs['self'] if past_kvs else None
        cross_kv = past_kvs['cross'] if past_kvs else None
        
        # 1. Causal Self-Attention
        norm_x = self.norm1(x)
        attn_out, new_self_kv = self.self_attn(
            norm_x, norm_x, freqs_cis_q=freqs_cis_dec, freqs_cis_k=freqs_cis_dec, is_causal=True, past_kv=self_kv
        )
        h = x + attn_out
        
        # 2. Cross-Attention
        norm_h = self.norm2(h)
        # Skip re-passing enc_out if we already have the cross-attention cached
        cross_input = None if cross_kv is not None else enc_out 
        cross_out, new_cross_kv = self.cross_attn(
            norm_h, cross_input, freqs_cis_q=freqs_cis_dec, freqs_cis_k=freqs_cis_enc, enc_mask=enc_mask, past_kv=cross_kv
        )
        h = h + cross_out
        
        # 3. FFN
        out = h + self.ffn(self.norm3(h))
        return out, {'self': new_self_kv, 'cross': new_cross_kv}

# ==========================================
# 4. Final Architecture
# ==========================================
class CustomAsymmetricLLM(torch_nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers=8, n_dec_layers=2, n_heads=8, n_kv_heads=4, max_seq_len=2048):
        super().__init__()
        self.embeddings = torch_nn.Embedding(vocab_size, d_model)
        hidden_dim = int(d_model * 8 / 3)
        
        self.enc_layers = torch_nn.ModuleList([EncoderBlock(d_model, n_heads, n_kv_heads, hidden_dim) for _ in range(n_enc_layers)])
        self.dec_layers = torch_nn.ModuleList([DecoderBlock(d_model, n_heads, n_kv_heads, hidden_dim) for _ in range(n_dec_layers)])
        
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        
        self.lm_head = torch_nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embeddings.weight 
        
        freqs_cis = precompute_freqs_cis(d_model // n_heads, max_seq_len * 2)
        self.register_buffer("freqs_cis", freqs_cis)

    def forward(self, enc_x, dec_x, enc_mask=None):
        """Standard Forward Pass for Training (No Caching)"""
        B, T_enc = enc_x.shape
        B, T_dec = dec_x.shape
        
        freqs_cis_enc = self.freqs_cis[:T_enc].to(enc_x.device)
        freqs_cis_dec = self.freqs_cis[:T_dec].to(dec_x.device)
        
        # CRITICAL FIX: Safely restore to complex on the GPU
        freqs_cis_enc = torch.view_as_complex(freqs_cis_enc)
        freqs_cis_dec = torch.view_as_complex(freqs_cis_dec)
        
        h_enc = self.embeddings(enc_x)
        for layer in self.enc_layers:
            h_enc = layer(h_enc, freqs_cis_enc, enc_mask)
        h_enc = self.enc_norm(h_enc)
        
        h_dec = self.embeddings(dec_x)
        for layer in self.dec_layers:
            h_dec, _ = layer(h_dec, h_enc, freqs_cis_dec, freqs_cis_enc, enc_mask, past_kvs=None)
        h_dec = self.dec_norm(h_dec)
        
        return self.lm_head(h_dec)

    @torch.no_grad()
    def generate(self, enc_x, enc_mask, max_new_tokens, temperature=1.0, top_p=1.0, bos_token_id=1):
        """Extremely fast Autoregressive Generation with KV Caching"""
        B = enc_x.size(0)
        device = enc_x.device
        
        # 1. Run the massive Encoder once
        T_enc = enc_x.shape[1]
        freqs_cis_enc = self.freqs_cis[:T_enc].to(device)
        freqs_cis_enc = torch.view_as_complex(freqs_cis_enc) # Restore to complex
        
        h_enc = self.embeddings(enc_x)
        for layer in self.enc_layers:
            h_enc = layer(h_enc, freqs_cis_enc, enc_mask)
        h_enc = self.enc_norm(h_enc)
        
        # 2. Initialize sequence and empty cache state
        dec_x = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        past_kvs = [None for _ in range(len(self.dec_layers))]
        
        for step in range(max_new_tokens):
            # Only process the single newest token!
            current_token = dec_x[:, -1:] 
            freqs_cis_dec = self.freqs_cis[step : step+1].to(device)
            freqs_cis_dec = torch.view_as_complex(freqs_cis_dec) # Restore to complex
            
            h_dec = self.embeddings(current_token)
            new_past_kvs = []
            
            for i, layer in enumerate(self.dec_layers):
                h_dec, layer_kvs = layer(
                    h_dec, h_enc, freqs_cis_dec, freqs_cis_enc, enc_mask, past_kvs=past_kvs[i]
                )
                new_past_kvs.append(layer_kvs)
                
            past_kvs = new_past_kvs # Update cache for the next loop
            h_dec = self.dec_norm(h_dec)
            
            logits = self.lm_head(h_dec[:, -1, :])
            
            if temperature > 0:
                logits = logits / temperature
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    for i in range(B):
                        indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                        logits[i, indices_to_remove] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
                
            dec_x = torch.cat((dec_x, next_token), dim=1)
            
        return dec_x[:, 1:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Split your total N_LAYER (15) asymmetrically for the Encoder-Decoder
ENC_LAYERS = 12
DEC_LAYERS = 3

# Use explicit keyword arguments to prevent positional shifting!
model = CustomAsymmetricLLM(
    vocab_size=VOCAB_SIZE, 
    d_model=N_EMBD, 
    n_enc_layers=ENC_LAYERS, 
    n_dec_layers=DEC_LAYERS, 
    n_heads=N_HEAD, 
    n_kv_heads=N_KV_HEAD, 
    max_seq_len=MODEL_CONTEXT_LENGTH
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

model.to(device)
if torch.cuda.device_count() > 1:
    print(f"Maximizing compute: Using {torch.cuda.device_count()} GPUs with DataParallel.")
    # Use torch.nn explicitly to avoid import alias errors
    model = torch.nn.DataParallel(model)

Total trainable parameters: 62,007,296
Maximizing compute: Using 2 GPUs with DataParallel.


In [9]:
# Define the path for the plot since it was missing
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

# ==========================================
# 5. Training Loop with Resume & Plotting (Encoder-Decoder)
# ==========================================
from torch.optim.lr_scheduler import OneCycleLR

optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
scaler = GradScaler('cuda')

# Safe buffer prevents the OneCycleLR ValueError on the final batch
total_steps = ((len(train_loader) + ACCUMULATION_STEPS - 1) // ACCUMULATION_STEPS) * EPOCHS
scheduler = OneCycleLR(
    optimizer, 
    max_lr=5e-4, 
    total_steps=total_steps + 10, 
    pct_start=0.1, 
    anneal_strategy='cos'
)

start_epoch = 0
global_step = 0
resume_batch_idx = -1
train_steps, train_loss, val_steps, val_loss = [], [], [], []
LOGGING_INTERVAL = 100

if os.path.exists(CHECKPOINT_PATH):
    print(f"Resuming from checkpoint: {CHECKPOINT_PATH}")
    checkpoint_data = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(checkpoint_data['model_state_dict'])
    else:
        model.load_state_dict(checkpoint_data['model_state_dict'])
        
    optimizer.load_state_dict(checkpoint_data['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint_data['scaler_state_dict'])
    
    if 'scheduler_state_dict' in checkpoint_data:
        scheduler.load_state_dict(checkpoint_data['scheduler_state_dict'])
        
    start_epoch = checkpoint_data['epoch']
    global_step = checkpoint_data['global_step']
    resume_batch_idx = checkpoint_data.get('batch_idx', -1)
    train_loss = checkpoint_data.get('train_loss', [])
    val_loss = checkpoint_data.get('val_loss', [])
    val_steps = checkpoint_data.get('val_steps', [])
    train_steps = checkpoint_data.get('train_steps', [])
    
val_loader_iter = iter(val_loader)
print("Starting Training...")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    running_loss = 0.0  
    
    for i, (x, y) in enumerate(progress_bar):
        if epoch == start_epoch and i <= resume_batch_idx:
            continue
            
        x, y = x.to(device), y.to(device)

        # Convert sequence to Seq2Seq task using the full remaining length
        split_idx = x.size(1) // 2
        enc_x = x[:, :split_idx]        
        dec_x = x[:, split_idx:]      
        y_target = y[:, split_idx:]   
        
        with autocast('cuda', dtype=torch.float16):
            # Model takes two inputs, returns one output
            logits = model(enc_x, dec_x)
            # Use .reshape() instead of .view() to handle non-contiguous sliced memory safely
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y_target.reshape(-1)) / ACCUMULATION_STEPS
            
        scaler.scale(loss).backward()
        
        running_loss += loss.item() * ACCUMULATION_STEPS
        is_last_batch = (i + 1) == len(train_loader)
        
        if (i + 1) % ACCUMULATION_STEPS == 0 or is_last_batch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            try:
                scheduler.step()
            except ValueError:
                pass
                
            optimizer.zero_grad()
            global_step += 1
            
            steps_in_cycle = ACCUMULATION_STEPS if not is_last_batch else ((i + 1) % ACCUMULATION_STEPS or ACCUMULATION_STEPS)
            current_train_loss = running_loss / steps_in_cycle
            running_loss = 0.0  
            
            if global_step % LOGGING_INTERVAL == 0:
                train_steps.append(global_step)
                train_loss.append(current_train_loss)
                
                model.eval()
                with torch.no_grad():
                    val_loss_accum = 0.0
                    val_eval_steps = 10  
                    
                    for _ in range(val_eval_steps):
                        try:
                            val_x, val_y = next(val_loader_iter)
                        except StopIteration:
                            val_loader_iter = iter(val_loader)
                            val_x, val_y = next(val_loader_iter)
                            
                        val_x, val_y = val_x.to(device), val_y.to(device)
                        
                        # Apply identical Seq2Seq split to validation data
                        v_split_idx = val_x.size(1) // 2
                        enc_val_x = val_x[:, :v_split_idx]
                        dec_val_x = val_x[:, v_split_idx:]
                        val_y_target = val_y[:, v_split_idx:]
                        
                        with autocast('cuda', dtype=torch.float16):
                            v_logits = model(enc_val_x, dec_val_x)
                            v_loss = F.cross_entropy(v_logits.reshape(-1, v_logits.size(-1)), val_y_target.reshape(-1))
                            val_loss_accum += v_loss.item()
                            
                    avg_v_loss = val_loss_accum / val_eval_steps
                    
                val_steps.append(global_step)
                val_loss.append(avg_v_loss)
                
                model.train()
                current_lr = scheduler.get_last_lr()[0]
                progress_bar.set_postfix({
                    "LR": f"{current_lr:.2e}",
                    "Train Loss": f"{current_train_loss:.4f}", 
                    "Val Loss": f"{avg_v_loss:.4f}"
                })
                
            if global_step % 500 == 0:
                state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                
                torch.save({
                    'epoch': epoch,
                    'batch_idx': i,
                    'global_step': global_step,
                    'model_state_dict': state_dict,
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scaler_state_dict': scaler.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'train_steps': train_steps,
                    'train_loss': train_loss,
                    'val_steps': val_steps,
                    'val_loss': val_loss
                }, CHECKPOINT_PATH)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
final_state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save(final_state_dict, os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin"))
print("Training completed successfully!")

Resuming from checkpoint: /kaggle/working/cs781_phase1/latest_checkpoint.pt
Starting Training...



Epoch 1: 100%|██████████| 24414/24414 [01:26<00:00, 281.06it/s][A


Training completed successfully!


In [ ]:
# def generate_batch(prompts, model, tokenizer, device, target_words=128):
#     inputs = tokenizer(
#         prompts,
#         return_tensors="pt",
#         padding=True,
#         truncation=True,
#         max_length=MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
#     )

#     # 1. Extract inputs and move to device
#     enc_x = inputs["input_ids"].to(device)
#     enc_mask_raw = inputs["attention_mask"].to(device)
    
#     # 2. Convert standard 1D padding mask (B, T) to 4D attention mask (B, 1, 1, T)
#     B, T = enc_mask_raw.shape
#     enc_mask = enc_mask_raw.view(B, 1, 1, T)
#     enc_mask = enc_mask.masked_fill(enc_mask == 0, float('-inf')).masked_fill(enc_mask == 1, 0.0)

#     # Use BOS token if available, otherwise EOS token as the starting sequence
#     bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id

#     # 3. Run Custom Generation
#     with torch.inference_mode():
#         with autocast('cuda', dtype=torch.float16):
#             generated_tokens = model.generate(
#                 enc_x=enc_x,
#                 enc_mask=enc_mask,
#                 max_new_tokens=MAX_GENERATION_TOKENS,
#                 temperature=TEMPERATURE,
#                 top_p=TOP_P,
#                 bos_token_id=bos_id
#             )

#     # 4. Decode and Trim
#     predictions = []
#     for i in range(len(prompts)):
#         generated_text = tokenizer.decode(generated_tokens[i], skip_special_tokens=True)
#         generated_words = generated_text.split()[:target_words]
#         predictions.append(" ".join(generated_words))

#     return predictions

In [ ]:
# # --- Batch Processing Loop ---
# predictions = {}
# GENERATION_BATCH_SIZE = 32

# items = list(test_data.items())
# total_examples = len(items)
# print(f"Total test examples: {total_examples}")

# for start in range(0, total_examples, GENERATION_BATCH_SIZE):
#     batch = items[start:start + GENERATION_BATCH_SIZE]
#     batch_ids = [str(example_id) for example_id, _ in batch]
#     batch_prompts = [record["text"] for _, record in batch]

#     batch_predictions = generate_batch(
#         prompts=batch_prompts,
#         model=eval_model, # Remember to use the unwrapped model from DataParallel!
#         tokenizer=tokenizer,
#         device=device,
#         target_words=128
#     )

#     for example_id, prediction in zip(batch_ids, batch_predictions):
#         predictions[example_id] = {"text": prediction}

#     completed = start + len(batch)
#     print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

# # Save and Zip
# PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
# SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

# with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
#     json.dump(predictions, f, ensure_ascii=False, indent=2)

# with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
#     zipf.write(PREDICTION_PATH, arcname="prediction.json")

# print("Codabench submission created:", SUBMISSION_ZIP)

In [13]:
import os
import json
import zipfile
import torch
from torch.amp import autocast
from tqdm import tqdm

# 1. Define the function FIRST so Python knows it exists
def generate_batch(prompts, model, tokenizer, device, target_words=128):
    batch_size = len(prompts)
    
    # 2. Use the CustomScratchBPE .encode() method
    encoded_prompts = [tokenizer.encode(p)[-MODEL_CONTEXT_LENGTH:] for p in prompts]
    max_len = max(len(p) for p in encoded_prompts)
    
    # Extract special tokens safely from your custom vocab dictionary
    pad_id = tokenizer.vocab.get("[PAD]", 1) 
    bos_id = tokenizer.vocab.get("[BOS]", 2) 
    
    # Manual Left-Padding
    padded_inputs = []
    for p in encoded_prompts:
        pad_len = max_len - len(p)
        padded_inputs.append([pad_id] * pad_len + p)
        
    enc_x = torch.tensor(padded_inputs, dtype=torch.long).to(device)
    
    # Create the 4D enc_mask for your CustomAsymmetricLLM
    # Create the 4D enc_mask for your CustomAsymmetricLLM
    # CRITICAL FIX: Cast to .float() instead of .long() so it can hold '-inf'
    enc_mask_raw = (enc_x != pad_id).float().to(device)
    B, T = enc_mask_raw.shape
    enc_mask = enc_mask_raw.view(B, 1, 1, T)
    enc_mask = enc_mask.masked_fill(enc_mask == 0.0, float('-inf')).masked_fill(enc_mask == 1.0, 0.0)

    # 3. Run Custom Generation
    with torch.inference_mode():
        with autocast('cuda', dtype=torch.float16):
            generated_tokens = model.generate(
                enc_x=enc_x,
                enc_mask=enc_mask,
                max_new_tokens=MAX_GENERATION_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                bos_token_id=bos_id
            )

    # 4. Decode and Trim
    predictions = []
    for i in range(batch_size):
        # Decode the newly generated tokens (output_tokens from the generate method excludes the prompt)
        generated_text = tokenizer.decode(generated_tokens[i].tolist())
        generated_words = generated_text.split()[:target_words]
        predictions.append(" ".join(generated_words))

    return predictions


# ==========================================
# --- Extract Test Data from Local Disk ---
# ==========================================
print("Downloading Test file...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_1_Pretraining/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
# ==========================================
# --- Execution Loop ---
# ==========================================
predictions = {}

# WARNING: Keep batch size extremely small (1 to 4) for the Encoder-Decoder to avoid VRAM crashes
GENERATION_BATCH_SIZE = 2 

items = list(test_data.items())
total_examples = len(items)
print(f"Total test examples: {total_examples}")

# Extract inner model if wrapped in DataParallel
eval_model = model.module if isinstance(model, torch.nn.DataParallel) else model
eval_model.eval()

# Use tqdm for a clean progress bar
for start in tqdm(range(0, total_examples, GENERATION_BATCH_SIZE), desc="Generating Batches"):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    # Safely call the function now that it is defined above
    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=eval_model, 
        tokenizer=tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS
    )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

# Save and Zip
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:", SUBMISSION_ZIP)

Total test examples: 1000



Generating Batches: 100%|██████████| 500/500 [18:15<00:00,  2.19s/it]

Codabench submission created: /kaggle/working/cs781_phase1/submission.zip
